In [0]:
catalog = "job_market"
schema = "bronze"
landing_path = "/Volumes/job_market/landing/raw_files/"

In [0]:

from pyspark.sql import functions as F

# 1. Parquet-Dateien mit Auto Loader lesen
def read_files(source_path, checkpoint_path):
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", checkpoint_path)
        .load(source_path)
    )


# 2. Metadaten hinzufügen
def add_metadata(df):
    return (
        df.withColumn(
            "source_file",
            F.col("_metadata.file_path")
        )
        .withColumn(
            "ingested_at",
            F.current_timestamp()
        )
    )


# 3. Streaming DataFrame als Delta speichern
def write_files(df, table_name, checkpoint_path):
    query = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(table_name)
    )

    query.awaitTermination()
